In [1]:
import numpy as np
import pandas as pd
from typing import Tuple
from datetime import datetime, timedelta

# Pull Simulation

In [2]:
# Probability functions for character and lightcone banners (separate systems)

def character_probability(pity):
    """
    Return the 5-star character probability for the current pity.
    Base chance is 0.6% for pulls 0-72; soft pity increases linearly from pull 73 to 90;
    Hard pity at pull 90 yields a guaranteed 5-star.

    Args:
        pity (int): The current pity count (number of pulls since last 5-star).

    Returns:
        float: The probability of pulling a 5-star on the next pull.

    """
    if pity < 73:
        return 0.006
    elif pity < 90:
        # Linear increase from 0.006 at pity = 73 to ~1 at pity = 90.
        increment = 0.05847  # approximate increment per pull in soft pity range.
        return min(1.0, 0.006 + (pity - 73 + 1) * increment)
    else:
        return 1.0

def lightcone_probability(pity):
    """
    Return the 5-star Light Cone probability for the current LC pity.
    Base chance is 0.8% for pulls 0-64; soft pity increases linearly from pull 65 to 80;
    Hard pity at pull 80 is a guaranteed 5-star.

    Args:
        pity (int): The current LC pity count (number of pulls since last 5-star).

    Returns:
        float: The probability of pulling a 5-star on the next LC pull.
    """
    if pity < 65:
        return 0.008
    elif pity < 80:
        increment = 0.07  # approximate increment per pull in soft pity range.
        return min(1.0, 0.008 + (pity - 65 + 1) * increment)
    else:
        return 1.0

In [3]:
# Define pull simulation logic with the user's current situation

def simulate_combo(
    total_pulls: int,
    desired_chars: int,
    desired_lcs: int,
    start_char_pity: int = 0,
    start_char_guarantee: bool = False,
    start_lc_pity: int = 0,
    start_lc_guarantee: bool = False,
    full_4star_chars: bool = True,
    simulate_4star: bool = True
) -> Tuple[bool, int, float, float]:
    """
    Simulates pulls to reach desired characters and lightcones.
    Tracks refunded pulls and leftover pulls after simulation.

    Args:
        total_pulls (int): Total available pulls.
        desired_chars (int): Number of featured characters to obtain.
        desired_lcs (int): Number of featured lightcones to obtain.
        start_char_pity (int, optional): Pity count for the character banner. Defaults to 0.
        start_char_guarantee (bool, optional): Whether the next character is guaranteed. Defaults to False.
        start_lc_pity (int, optional): Pity count for the lightcone banner. Defaults to 0.
        start_lc_guarantee (bool, optional): Whether the next lightcone is guaranteed. Defaults to False.
        full_4star_chars (bool, optional): If True, refunds are granted for 4★ character duplicates. Defaults to True.
        simulate_4star (bool, optional): Whether to simulate 4★ pull refunds. Defaults to True.

    Returns:
        tuple[bool, int, int]:
            - success (bool): Whether all desired items were obtained.
            - pulls_used (int): Number of pulls used in the simulation.
            - leftover_pulls (int): Number of pulls remaining (including refunds).
    """
    remaining = total_pulls
    used_pulls = 0
    refunded_pulls = 0.0
    chars_obtained = 0
    lcs_obtained = 0

    pity_char, guarantee_char = start_char_pity, start_char_guarantee
    pity_lc, guarantee_lc = start_lc_pity, start_lc_guarantee
    pity_4star_char = 0
    pity_4star_lc = 0

    while remaining > 0 and (chars_obtained < desired_chars or lcs_obtained < desired_lcs):
        # Character pulls
        while chars_obtained < desired_chars and remaining > 0:
            remaining -= 1
            used_pulls += 1
            pity_char += 1
            pity_4star_char += 1

            hit_5star = np.random.rand() < character_probability(pity_char)

            if hit_5star:
                if guarantee_char or np.random.rand() < 0.5:
                    chars_obtained += 1
                    guarantee_char = False
                else:
                    guarantee_char = True
                pity_char = 0
                pity_4star_char = 0
                break  # restart loop to re-check char/LC goals

            # 4-star refund simulation (if no 5-star hit)
            if simulate_4star and (pity_4star_char >= 10 or np.random.rand() < 0.051):
                pity_4star_char = 0
                if full_4star_chars:
                    remaining += 1
                    refunded_pulls += 1.0

        # Light cone pulls
        while lcs_obtained < desired_lcs and remaining > 0:
            remaining -= 1
            used_pulls += 1
            pity_lc += 1
            pity_4star_lc += 1

            hit_5star = np.random.rand() < lightcone_probability(pity_lc)

            if hit_5star:
                if guarantee_lc or np.random.rand() < 0.75:
                    lcs_obtained += 1
                    guarantee_lc = False
                else:
                    guarantee_lc = True
                pity_lc = 0
                pity_4star_lc = 0
                break  # restart loop to re-check char/LC goals

            # 4-star refund simulation (if no 5-star hit)
            if simulate_4star and (pity_4star_lc >= 10 or np.random.rand() < 0.06):
                pity_4star_lc = 0
                remaining += 0.4
                refunded_pulls += 0.4

    success = (chars_obtained >= desired_chars and lcs_obtained >= desired_lcs)
    pulls_leftover = remaining
    return success, used_pulls, round(refunded_pulls, 2), round(pulls_leftover, 2)

In [4]:
def run_simulation(
    total_pulls,
    desired_chars,
    desired_lcs,
    start_char_pity=0,
    start_char_guarantee=False,
    start_lc_pity=0,
    start_lc_guarantee=False,
    trials=10000
):

    """
    Runs a Monte Carlo simulation over multiple trials to estimate probability of
    success and average pulls used for obtaining the desired units.

    Args:
        inputs (BasicSimulationInput): Input dictionary with simulation configuration:
            - total_pulls
            - desired_chars
            - desired_lcs
            - start_char_pity
            - start_char_guarantee
            - start_lc_pity
            - start_lc_guarantee
            - trials

    Returns:
        tuple[float, float, float]:
            - Probability of success across trials.
            - Average number of pulls used.
            - Average number of pulls left over (including refunds).
    """

    successes = 0
    pulls_used = []
    pulls_refunded = []
    pulls_leftover = []

    all_used = []
    all_refunded = []
    all_leftover = []

    for _ in range(trials):
        success, used, refunded, leftover = simulate_combo(
            total_pulls,
            desired_chars,
            desired_lcs,
            start_char_pity,
            start_char_guarantee,
            start_lc_pity,
            start_lc_guarantee
        )

        all_used.append(used)
        all_refunded.append(refunded)
        all_leftover.append(leftover)

        if success:
            successes += 1
            pulls_used.append(used)
            pulls_refunded.append(refunded)
            pulls_leftover.append(leftover)

    failures = trials - successes

    success_rate = successes / trials
    avg_pulls_success = np.mean(pulls_used) if pulls_used else 0
    avg_refund_success = np.mean(pulls_refunded) if pulls_refunded else 0
    avg_left_success = np.mean(pulls_leftover) if pulls_leftover else 0
    avg_pulls_all = np.mean(all_used)
    avg_left_all = np.mean(all_leftover)

    fail_indices = [i for i in range(trials) if i >= len(pulls_used)]
    fail_pulls = [all_used[i] for i in fail_indices]
    fail_refunds = [all_refunded[i] for i in fail_indices]
    fail_leftover = [all_leftover[i] for i in fail_indices]

    avg_pulls_fail = np.mean(fail_pulls) if fail_pulls else 0
    avg_refund_fail = np.mean(fail_refunds) if fail_refunds else 0
    avg_left_fail = np.mean(fail_leftover) if fail_leftover else 0

    return (
        success_rate,
        avg_pulls_success,
        avg_refund_success,
        avg_left_success,
        avg_pulls_all,
        avg_left_all,
        failures,
        avg_pulls_fail,
        avg_refund_fail,
        avg_left_fail
    )

## Pull Simulation Test

In [5]:
scenarios = {
    "E2S1": {"desired_chars": 3, "desired_lcs": 1, "avail_pulls": 183}
}

for name, params in scenarios.items():
    (
        success_rate,
        avg_pulls_success,
        avg_refund_success,
        avg_left_success,
        avg_pulls_all,
        avg_left_all,
        failures,
        avg_pulls_fail,
        avg_refund_fail,
        avg_left_fail
    ) = run_simulation(
        total_pulls=params['avail_pulls'],
        desired_chars=params["desired_chars"],
        desired_lcs=params["desired_lcs"],
        start_char_pity=3,
        start_char_guarantee=False,
        start_lc_pity=10,
        start_lc_guarantee=False,
        trials=50000
    )

    print(f"\n{name} Simulation Results (50,000 Trials on {params['avail_pulls']} pulls):")
    print(f"  • Success Rate:              {success_rate * 100:.2f}%")
    print(f"  • Failures:                  {failures:,}")
    print(f"  --- Success Cases ---")
    print(f"  • Avg Pulls Used:            {avg_pulls_success:.2f}")
    print(f"  • Avg Refunds Earned:        {avg_refund_success:.2f}")
    print(f"  • Avg Pulls Left Over:       {avg_left_success:.2f}")
    print(f"  --- Failure Cases ---")
    print(f"  • Avg Pulls Used:            {avg_pulls_fail:.2f}")
    print(f"  • Avg Refunds Earned:        {avg_refund_fail:.2f}")
    print(f"  • Avg Pulls Left Over:       {avg_left_fail:.2f}")
    print(f"  --- Overall (All Trials) ---")
    print(f"  • Avg Pulls Used:            {avg_pulls_all:.2f}")
    print(f"  • Avg Pulls Left Over:       {avg_left_all:.2f}")


E2S1 Simulation Results (50,000 Trials on 183 pulls):
  • Success Rate:              4.85%
  • Failures:                  47,573
  --- Success Cases ---
  • Avg Pulls Used:            174.00
  • Avg Refunds Earned:        17.39
  • Avg Pulls Left Over:       26.39
  --- Failure Cases ---
  • Avg Pulls Used:            202.10
  • Avg Refunds Earned:        19.89
  • Avg Pulls Left Over:       0.79
  --- Overall (All Trials) ---
  • Avg Pulls Used:            202.11
  • Avg Pulls Left Over:       0.78


### In-Progress: Additional Insight Into Simulation

In [6]:
def simulate_combo_verbose(
    total_pulls: int,
    desired_chars: int,
    desired_lcs: int,
    start_char_pity: int = 0,
    start_char_guarantee: bool = False,
    start_lc_pity: int = 0,
    start_lc_guarantee: bool = False,
    full_4star_chars: bool = True,
    simulate_4star: bool = True
) -> Tuple[bool, int, float, float, dict]:
    """
    Simulates pulls with metadata tracking for verbose analysis.

    Returns:
        - success (bool)
        - pulls_used (int)
        - refunded (float)
        - leftover (float)
        - metadata (dict)
    """
    remaining = total_pulls
    used_pulls = 0
    refunded_pulls = 0.0
    chars_obtained = 0
    lcs_obtained = 0

    pity_char, guarantee_char = start_char_pity, start_char_guarantee
    pity_lc, guarantee_lc = start_lc_pity, start_lc_guarantee
    pity_4star_char = 0
    pity_4star_lc = 0

    char_50_50_wins = 0
    lc_75_25_wins = 0
    char_50_50_encounters = 0  # Track 50/50 encounters
    lc_75_25_encounters = 0  # Track 75/25 encounters
    lost_char_50_50_failure = False
    pity_char_trigger = 0
    pity_lc_trigger = 0

    def character_probability(pity):
        if pity < 73: return 0.006
        if pity < 90: return min(1.0, 0.006 + (pity - 72) * 0.05847)
        return 1.0

    def lightcone_probability(pity):
        if pity < 65: return 0.008
        if pity < 80: return min(1.0, 0.008 + (pity - 64) * 0.07)
        return 1.0

    while remaining > 0 and (chars_obtained < desired_chars or lcs_obtained < desired_lcs):
        while chars_obtained < desired_chars and remaining > 0:
            remaining -= 1
            used_pulls += 1
            pity_char += 1
            pity_4star_char += 1

            hit_5star = np.random.rand() < character_probability(pity_char)

            if hit_5star:
                pity_char_trigger = pity_char
                char_50_50_encounters += 1  # Increment 50/50 encounter

                if guarantee_char or np.random.rand() < 0.5:
                    chars_obtained += 1
                    guarantee_char = False
                    char_50_50_wins += 1
                else:
                    guarantee_char = True
                pity_char = 0
                pity_4star_char = 0
                break

            if simulate_4star and (pity_4star_char >= 10 or np.random.rand() < 0.051):
                pity_4star_char = 0
                if full_4star_chars:
                    remaining += 1
                    refunded_pulls += 1.0

        while lcs_obtained < desired_lcs and remaining > 0:
            remaining -= 1
            used_pulls += 1
            pity_lc += 1
            pity_4star_lc += 1

            hit_5star = np.random.rand() < lightcone_probability(pity_lc)

            if hit_5star:
                pity_lc_trigger = pity_lc
                lc_75_25_encounters += 1  # Increment 75/25 encounter

                if guarantee_lc or np.random.rand() < 0.75:
                    lcs_obtained += 1
                    guarantee_lc = False
                    lc_75_25_wins += 1
                else:
                    guarantee_lc = True
                pity_lc = 0
                pity_4star_lc = 0
                break

            if simulate_4star and (pity_4star_lc >= 10 or np.random.rand() < 0.06):
                pity_4star_lc = 0
                remaining += 0.4
                refunded_pulls += 0.4

    success = (chars_obtained >= desired_chars and lcs_obtained >= desired_lcs)
    pulls_leftover = remaining

    # Capture the final pity values
    final_pity_char = pity_char # Store final character pity
    final_pity_lc = pity_lc # Store final light cone pity


    if not success and guarantee_char:  # lost 50/50 and ran out
        lost_char_50_50_failure = True

    meta = {
        "char_50_50_wins": char_50_50_wins,
        "lc_75_25_wins": lc_75_25_wins,
        "pity_char_trigger": pity_char_trigger,
        "pity_lc_trigger": pity_lc_trigger,
        "lost_char_50_50_failure": int(lost_char_50_50_failure),
        "final_pity_char": final_pity_char,
        "final_pity_lc": final_pity_lc,
        "char_50_50_encounters": char_50_50_encounters,  # Include 50/50 encounters
        "lc_75_25_encounters": lc_75_25_encounters      # Include 75/25 encounters
    }

    return success, used_pulls, round(refunded_pulls, 2), round(pulls_leftover, 2), meta


In [7]:
def run_simulation_verbose(
    total_pulls,
    desired_chars,
    desired_lcs,
    start_char_pity=0,
    start_char_guarantee=False,
    start_lc_pity=0,
    start_lc_guarantee=False,
    trials=10000
):
    """
    Verbose Monte Carlo simulation with metadata aggregation and full printed summary.

    Returns:
        Tuple of:
        - success_rate (float)
        - avg_pulls_success (float)
        - avg_refund_success (float)
        - avg_left_success (float)
        - avg_pulls_all (float)
        - avg_left_all (float)
        - failures (int)
        - avg_pulls_fail (float)
        - avg_refund_fail (float)
        - avg_left_fail (float)
        - stats_summary (dict)
    """
    successes = 0
    pulls_used = []
    pulls_refunded = []
    pulls_leftover = []

    all_used = []
    all_refunded = []
    all_leftover = []

    # Track failure cases for pulls used and refunds
    fail_pulls = []
    fail_refunds = []
    fail_leftover = []

    metadata_logs = []

    for _ in range(trials):
        success, used, refunded, leftover, meta = simulate_combo_verbose(
            total_pulls,
            desired_chars,
            desired_lcs,
            start_char_pity,
            start_char_guarantee,
            start_lc_pity,
            start_lc_guarantee
        )

        all_used.append(used)
        all_refunded.append(refunded)
        all_leftover.append(leftover)
        metadata_logs.append({**meta, "success": success})

        if success:
            successes += 1
            pulls_used.append(used)
            pulls_refunded.append(refunded)
            pulls_leftover.append(leftover)

        else:
            fail_pulls.append(used)
            fail_refunds.append(refunded)

    failures = trials - successes
    success_rate = successes / trials

    df = pd.DataFrame(metadata_logs)

    # Overall stats
    avg_pulls_all = np.mean(all_used)
    avg_left_all = np.mean(all_leftover)

    ### Success stats ###

    # Averages across all successful runs
    avg_pulls_success = np.mean(pulls_used) if pulls_used else 0
    avg_refund_success = np.mean(pulls_refunded) if pulls_refunded else 0
    avg_left_success = np.mean(pulls_leftover) if pulls_leftover else 0

    # Updated: Calculate average pity for char and LC on success using DataFrame
    avg_pity_char_success = round(df[df.success]["pity_char_trigger"].mean(), 2) if not df[df.success].empty else None
    avg_pity_lc_success = round(df[df.success]["pity_lc_trigger"].mean(), 2) if not df[df.success].empty else None

    refund_rate_success = round(np.mean(pulls_refunded) / np.mean(pulls_used) if pulls_used else 0, 4)

    # Updated: Correctly calculate 50/50 and 75/25 win rates for successes

    # Sum the char_50_50_encounters and lc_75_25_encounters for successful runs to get accurate totals
    total_50_50s_encountered_in_successes = df[df.success]["char_50_50_encounters"].sum()
    total_25_75s_encountered_in_successes = df[df.success]["lc_75_25_encounters"].sum()

    won_50_50_runs_successes = df[(df.success) & (df.char_50_50_wins > 0)]["char_50_50_wins"].sum()
    won_25_75_runs_successes = df[(df.success) & (df.lc_75_25_wins > 0)]["lc_75_25_wins"].sum()

    successes_char_win_rate = f"{((won_50_50_runs_successes / total_50_50s_encountered_in_successes) if total_50_50s_encountered_in_successes > 0 else np.nan) * 100:.2f}%"
    successes_lc_win_rate = f"{((won_25_75_runs_successes / total_25_75s_encountered_in_successes) if total_25_75s_encountered_in_successes > 0 else np.nan) * 100:.2f}%"

    # ... (Existing win rate calculations for failures) ...

    ### Failure stats ###
    # Updated: Directly use fail_pulls list for calculating average pulls on failure
    avg_pulls_fail = np.mean(fail_pulls) if fail_pulls else 0
    avg_refund_fail = np.mean(fail_refunds) if fail_refunds else 0
    avg_left_fail = np.mean(fail_leftover) if fail_leftover else 0

    # Updated: Calculate average pity for char and LC on failure using DataFrame
    avg_pity_char_failure = round(df[~df.success]["final_pity_char"].mean(), 2) if not df[~df.success].empty else None
    avg_pity_lc_failure = round(df[~df.success]["final_pity_lc"].mean(), 2) if not df[~df.success].empty else None

    refund_rate_failure = round(np.mean(fail_refunds) / np.mean(fail_pulls) if fail_pulls else 0, 4)

    #Updated: Use .sum() on char_50_50_wins to get total 50/50 encounters in failed runs
    # Sum the char_50_50_encounters and lc_75_25_encounters for failed runs to get accurate totals
    total_50_50s_encountered_in_failures = df[~df.success]["char_50_50_encounters"].sum()
    total_25_75s_encountered_in_failures = df[~df.success]["lc_75_25_encounters"].sum()

    won_50_50_runs_failures = df[(~df.success) & (df.char_50_50_wins > 0)]["char_50_50_wins"].sum()
    won_25_75_runs_failures = df[(~df.success) & (df.lc_75_25_wins > 0)]["lc_75_25_wins"].sum()

    failure_char_win_rate = f"{((won_50_50_runs_failures / total_50_50s_encountered_in_failures) if total_50_50s_encountered_in_failures > 0 else np.nan) * 100:.2f}%"
    failure_lc_win_rate = f"{((won_25_75_runs_failures / total_25_75s_encountered_in_failures) if total_25_75s_encountered_in_failures > 0 else np.nan) * 100:.2f}%"

    stats_summary = {
        # Initial scenario
        "initial_pulls": total_pulls,
        "desired_characters": desired_chars,
        "desired_lightcones": desired_lcs,
        "success_rate": f"{success_rate * 100:.2f}%",
        "start_char_pity": start_char_pity,
        "start_char_guarantee": start_char_guarantee,
        "start_lc_pity": start_lc_pity,
        "start_lc_guarantee": start_lc_guarantee,

        # Stats for successful runs
        "avg_pity_char": round(df[df.success]["pity_char_trigger"].mean(), 2) if not df[df.success].empty else None,
        "avg_pity_lc": round(df[df.success]["pity_lc_trigger"].mean(), 2) if not df[df.success].empty else None,
        "successes_char_win_rate": successes_char_win_rate,
        "successes_lc_win_rate": successes_lc_win_rate,
        "avg_leftover_pulls_on_success": round(avg_left_success, 2),
        "avg_refund_success": round(avg_refund_success,0),

        # Stats for failed runs
        "failure_char_win_rate": failure_char_win_rate,
        "failure_lc_win_rate": failure_lc_win_rate,
        "avg_leftover_pulls_on_failure": round(avg_left_fail, 2),
        "avg_refund_fail": round(avg_refund_fail,0),
    }

    print("\nDetailed Stats Across {trials:,} Trials".format(trials=trials))
    print(f"• Initial Pulls:                {stats_summary['initial_pulls']}")
    print(f"• Pull Goal:                E{desired_chars - 1}S{desired_lcs}")
    print(f"• Success Rate:                {stats_summary['success_rate']}")
    print(f"• Avg Pity for Character Wins (Success Cases):      {stats_summary['avg_pity_char']}")
    print(f"• Avg Pity for Light Cone Wins (Success Cases):     {stats_summary['avg_pity_lc']}")
    print(f"• 50/50 Win Rate (Success Cases):  {stats_summary['successes_char_win_rate']}")
    print(f"• 75/25 LC Win Rate (Success Cases):           {stats_summary['successes_lc_win_rate']}")
    print(f"• Avg Leftover Pulls (Success Cases): {stats_summary['avg_leftover_pulls_on_success']}")
    print(f"• Avg Refund (Success Cases): {stats_summary['avg_refund_success']}")
    print(f"• 50/50 Win Rate (Failure Cases):  {stats_summary['failure_char_win_rate']}")
    print(f"• 75/25 LC Win Rate (Failure Cases):           {stats_summary['failure_lc_win_rate']}")
    print(f"• Avg Leftover Pulls (Failure Cases): {stats_summary['avg_leftover_pulls_on_failure']}")
    print(f"• Avg Refund (Failure Cases): {stats_summary['avg_refund_fail']}")
    print(f"• Starting Character Pity:     {start_char_pity} | Guarantee: {start_char_guarantee}")
    print(f"• Starting Light Cone Pity:    {start_lc_pity} | Guarantee: {start_lc_guarantee}")

    return (
        success_rate,
        avg_pulls_success,
        avg_refund_success,
        avg_left_success,
        avg_pulls_all,
        avg_left_all,
        failures,
        avg_pulls_fail,
        avg_refund_fail,
        avg_left_fail,
        stats_summary
    )

In [9]:
import openai

# Read API key from uploaded file
with open("OpenAI.txt", "r") as f:
    api_key = f.read().strip()

In [10]:
def describe_goal(sim_stats):
    chars = sim_stats["desired_characters"]
    lcs = sim_stats["desired_lightcones"]

    goal_label = f"E{chars - 1}S{lcs}"
    char_phrase = (
        "one copy of the limited 5★ character"
        if chars == 1 else
        f"{chars} copies of the limited 5★ character"
    )
    lc_phrase = (
        "no signature light cone"
        if lcs == 0 else
        "one signature 5★ light cone"
        if lcs == 1 else
        f"{lcs} signature 5★ light cones"
    )

    return goal_label, f"{char_phrase} and {lc_phrase}"


In [11]:
def analyze_sim_result(api_key, sim_stats, trials=50000):
    import openai
    from openai import OpenAI

    client = OpenAI(api_key=api_key)
    goal_label, goal_description = describe_goal(sim_stats)

    system_prompt = (
        "You are a Honkai Star Rail pull simulation assistant who explains probability and gacha mechanics. "
        "Explain statistical outcomes clearly, concisely, and with emphasis on player decision-making. "
        "Assume the reader is already familiar with HSR's pity systems."
    )

    user_prompt = """
The simulation goal was to reach {goal_label}: {goal_description}.

The player started with:
- Initial Pulls: {initial_pulls}
- Character Pity: {start_char_pity} | Guarantee: {start_char_guarantee}
- LC Pity: {start_lc_pity} | Guarantee: {start_lc_guarantee}

Simulation ran for {trials:,} trials. Here are the results:
- Success Rate: {success_rate}
- Character 50/50 Wins for Successful Runs: {successes_char_win_rate}
- Lightcone 75/25 Wins for Successful Runs: {successes_lc_win_rate}
- Avg Leftover Pulls for Successful Runs: {avg_leftover_pulls_on_success}
- Avg Character Pity for Successful Runs: {avg_pity_char}
- Avg Light Cone Pity for Successful Runs: {avg_pity_lc}
- Avg Refund for Successful Runs: {avg_refund_success}

- Character 50/50 Wins for Failed Runs: {failure_char_win_rate}
- Lightcone 75/25 Wins for Failed Runs: {failure_lc_win_rate}
- Avg Leftover Pulls for Failed Runs: {avg_leftover_pulls_on_failure}
- Avg Refund for Failed Runs: {avg_refund_fail}


As succinctly as possible, interpret the outcome. Answer questions like: Why did the success rate end up where it did?
Does the number of character and lightcone wins compared to the desired number mean that the user HAD to win at least certain number of 50/50s?
How does the character win rate compare to the baseline 50% odds, and similarly for lightcone win rate comparing to the 75% odds; what does this mean?
Is early pity wins are necessary for the successful cases? How does it compare to the failure cases?
Consider logically how players generally will obtain a character around soft pity of 80 or a lightcone around soft pity of 70, and multiply those numbers
by the number of desired characters and lightcones and sum them; does that number exceed or stay within the number of pulls they have? For example, if their number of pulls is
just barely within that number, then you would form a hypothesis that they really have to win character 50/50 and lightcone 75/25, and if they don't, then earlies are necessary, so you would look at the
data holistically to see if it supports this hypothesis, and if not revise. Present the final hypothesis to the user.
For context, character hard pity is 90, and lightcone hard pity is 80 and if there is no guarantee, there is a 50/50 or a 75/25, where it is possible the player does not
win the character or lightcone and the pity resets and they have to pull more.
For context, refunds are based on receiving a 4-star character and exchanging it for another pull in the HSR store. Since 4-stars are guaranteed at
least every 10 pulls, a higher number of refunds must be considered in tandem with how many pulls the player used. Lower number of pulls but higher refunds
implies the user got lucky with obtaining 4-stars and therefore got refunds. Higher number of fulls proportionately increases the number of refunds.
Therefore, refunds MUST be considered alongside how many leftover pulls there were and the ratio of those two between successful and unsuccessful runs must
be calculated and compared to produce meaningful analysis.

Lastly, provide a summary of the outcomes all considered together and if success isn't as likely, a suggestion for how many pulls may provide a buffer.
Present this information as succinctly as possible, in plain English without using statistical jargon.
""".format(goal_label=goal_label, goal_description=goal_description, trials=trials, **sim_stats) # unpack sim_stats


    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": system_prompt.strip()},
            {"role": "user", "content": user_prompt.strip()}
        ]
    )

    return response.choices[0].message.content.strip()

In [12]:
scenarios = {
    "E2S1": {"desired_chars": 3, "desired_lcs": 1, "avail_pulls": 284}
}
# 284 projected number incl. buying wishful resin support pack
for name, params in scenarios.items():
    (
        success_rate,
        avg_pulls_success,
        avg_refund_success,
        avg_left_success,
        avg_pulls_all,
        avg_left_all,
        failures,
        avg_pulls_fail,
        avg_refund_fail,
        avg_left_fail,
        sim_stats
    ) = run_simulation_verbose(
        total_pulls=params["avail_pulls"],
        desired_chars=params["desired_chars"],
        desired_lcs=params["desired_lcs"],
        start_char_pity=1,
        start_char_guarantee=False,
        start_lc_pity=2,
        start_lc_guarantee=False,
        trials=50000
    )

    # Feed into GPT-based analyzer
    explanation = analyze_sim_result(api_key=api_key, sim_stats=sim_stats)
    print("\nSimulation Explanation:")
    print(explanation)


Detailed Stats Across 50,000 Trials
• Initial Pulls:                284
• Pull Goal:                E2S1
• Success Rate:                39.70%
• Avg Pity for Character Wins (Success Cases):      55.16
• Avg Pity for Light Cone Wins (Success Cases):     47.62
• 50/50 Win Rate (Success Cases):  76.40%
• 75/25 LC Win Rate (Success Cases):           88.43%
• Avg Leftover Pulls (Success Cases): 48.19
• Avg Refund (Success Cases): 27.0
• 50/50 Win Rate (Failure Cases):  54.22%
• 75/25 LC Win Rate (Failure Cases):           75.18%
• Avg Leftover Pulls (Failure Cases): 0
• Avg Refund (Failure Cases): 32.0
• Starting Character Pity:     1 | Guarantee: False
• Starting Light Cone Pity:    2 | Guarantee: False

Simulation Explanation:
In your simulation aiming to reach E2S1, the success rate was around 39.70%. This means out of your 50,000 trials, you reached this goal about 4 out of 10 times. 

### Key Reasons for Success Rate:

1. **Pull Allocation and Pity:**
   - You started with 284 pulls. 

# Jades/ Pull Estimation

In [2]:
# Helper functions for regularly-occcurring events

def add_hoyolab_checkin(all_events, start_dt, end_dt):

    """
    Appends HoYoLAB login reward milestones (3x per month) to the event list.

    Args:
        all_events (list): The master list of (date, event name, jade) tuples.
        start_dt (datetime): Start date of the projection.
        end_dt (datetime): End date of the projection.

    Returns:
        None: Modifies all_events in place.
    """

    cur = datetime(start_dt.year, start_dt.month, 1)
    checkin_labels = {4: "1st", 12: "2nd", 19: "3rd"}

    while cur <= end_dt:
        year, month = cur.year, cur.month
        for day in [4, 12, 19]:
            try:
                checkin_date = datetime(year, month, day)
                if start_dt <= checkin_date <= end_dt:
                    label = checkin_labels.get(day, f"{day}th")
                    event_name = f"HoYoLAB Check-in ({label} Reward)"
                    all_events.append((checkin_date.strftime("%Y-%m-%d"), event_name, 20))
            except ValueError:
                continue  # Skip invalid dates like Feb 30

        # Move to next month
        if month == 12:
            cur = datetime(year + 1, 1, 1)
        else:
            cur = datetime(year, month + 1, 1)

def add_monthly_embers_reset(all_events, start_dt, end_dt):
    """
    Adds a row to the calendar for each 1st of the month to account for the
    5 free pulls from Embers exchange.

    Args:
        all_events (list): Event list to modify.
        start_dt (datetime): Projection start date.
        end_dt (datetime): Projection end date.
    """

    cur = datetime(start_dt.year, start_dt.month, 1)
    while cur <= end_dt:
        all_events.append((cur.strftime("%Y-%m-%d"), "Embers Exchange Reset", 0))
        next_month = cur.month % 12 + 1
        next_year = cur.year + (cur.month // 12)
        cur = datetime(next_year, next_month, 1)

def add_express_pass_renewal(all_events, express_start_str, end_dt):
    """
    Adds express supply renewal events every 30 days, each giving 300 jades.

    Args:
        all_events (list): Event list to modify.
        first_renewal_dt (datetime): Start date of the express pass renewal.
        end_dt (datetime): End date of the projection period.
    """

    er = datetime.strptime(express_start_str, "%Y-%m-%d")
    while er <= end_dt:
        all_events.append((er.strftime("%Y-%m-%d"), "Express Pass Renewal", 300))
        er += timedelta(days=30)

def add_endgame_resets(all_events, end_dt):
    """
    Adds recurring endgame content resets every 42 days (MoC, PF, Apoc Shadow), each giving 800 jades.

    Args:
        all_events (list): Event list to modify.
        end_dt (datetime): End date of projection.
    """

    def add_reset(start_str, name):
        d = datetime.strptime(start_str, "%Y-%m-%d")
        while d <= end_dt:
            all_events.append((d.strftime("%Y-%m-%d"), f"{name} Reset", 800))
            d += timedelta(days=42)
    add_reset("2025-03-03", "Apocalyptic Shadow")
    add_reset("2025-03-17", "Pure Fiction")
    add_reset("2025-03-31", "Memory of Chaos")

def add_weekly_simulated_universe(all_events, start_dt, end_dt):
    """
    Adds a weekly reward of 225 jades to simulate Simulated Universe weekly reset.

    Args:
        all_events (list): Event list to modify.
        start_dt (datetime): Start of projection.
        end_dt (datetime): End of projection.
    """

    cur = start_dt
    # Align to the next Monday if start_dt isn't a Monday
    while cur.weekday() != 0:  # 0 = Monday
        cur += timedelta(days=1)
    while cur <= end_dt:
        all_events.append((cur.strftime("%Y-%m-%d"), "Simulated Universe Weekly Reward", 225))
        cur += timedelta(weeks=1)

def add_nameless_honor_rewards(all_events, start_dt, end_dt):
    """
    Adds the full reward schedule of Nameless Honor (premium battle pass), assuming user purchases it.
    Includes immediate reward + estimated pull drops based on weekly progression.

    Args:
        all_events (list): The event list to append to.
        start_dt (datetime): Start date of projection.
        end_dt (datetime): End date of projection.
    """

    nh_start = datetime(2023, 4, 26)  # First Nameless Honor pass start
    cycle_length = timedelta(weeks=6)

    while nh_start <= end_dt:
        if nh_start >= start_dt:
            all_events.append((nh_start.strftime("%Y-%m-%d"), "Nameless Honor Purchase Bonus", 680))
            all_events.append(((nh_start + timedelta(days=7)).strftime("%Y-%m-%d"), "Nameless Honor Reward (1 Pull)", 160))
            all_events.append(((nh_start + timedelta(days=17)).strftime("%Y-%m-%d"), "Nameless Honor Reward (2 Pulls)", 320))
            all_events.append(((nh_start + timedelta(days=21)).strftime("%Y-%m-%d"), "Nameless Honor Reward (1 Pull)", 160))
        nh_start += cycle_length

In [3]:
def generate_pull_projection(
    current_pulls: int,
    start_date: str,
    end_date: str,
    events: list,
    express_renewal_start: str,
    nameless_honor: bool = False
) -> pd.DataFrame:
    """
    Generates a pull projection calendar showing when and how many pulls the user will accumulate
    based on known events, recurring income, and passive sources, including weekly passive income summaries every 7 days from the start date.

    Args:
        inputs (PullCalendarInput): Dictionary with projection parameters:
            - current_pulls (int): Starting pulls.
            - start_date (str): Start of the projection period (YYYY-MM-DD).
            - end_date (str): End of the projection period (YYYY-MM-DD).
            - events (list): List of tuples with (date, event name, jade amount).
            - express_renewal_start (str): First date of express pass renewal.
            - nameless_honor (bool): Whether Nameless Honor is purchased.

    Returns:
        pd.DataFrame: Table with columns [Date, Event, Jade Gain, Pull Gain, Cumulative Pulls].
    """
    start_dt = datetime.strptime(start_date, "%Y-%m-%d")
    end_dt = datetime.strptime(end_date, "%Y-%m-%d")
    jade_per_pull = 160.0
    daily_jade = 150
    weekly_passive_days = 7
    weekly_passive_jade_amount = weekly_passive_days * daily_jade

    all_events_list = list(events)

    # Add recurring events
    add_hoyolab_checkin(all_events_list, start_dt, end_dt)
    add_monthly_embers_reset(all_events_list, start_dt, end_dt)
    add_express_pass_renewal(all_events_list, express_renewal_start, end_dt)
    add_endgame_resets(all_events_list, end_dt)
    add_weekly_simulated_universe(all_events_list, start_dt, end_dt)
    if nameless_honor:
        add_nameless_honor_rewards(all_events_list, start_dt, end_dt)

    # Add weekly passive income entries every 7 days from start_date
    current_passive_date = start_dt + timedelta(days=weekly_passive_days)
    while current_passive_date <= end_dt:
        all_events_list.append((
            current_passive_date.strftime("%Y-%m-%d"),
            f"Weekly Passive Income ({weekly_passive_days} days)",
            weekly_passive_jade_amount # Add the total jade for 7 days
        ))
        current_passive_date += timedelta(days=weekly_passive_days)


    # Sort all events by date
    all_events_list.sort(key=lambda x: x[0])

    # Group events by date and filter for events on or after start_date
    events_by_date = {}
    for date_str, name, jade in all_events_list:
        event_date = datetime.strptime(date_str, "%Y-%m-%d")
        if event_date >= start_dt: # Only include events on or after start_date
            if date_str not in events_by_date:
                events_by_date[date_str] = []
            events_by_date[date_str].append((name, jade))

    cumulative = float(current_pulls)
    rows = []
    current_date = start_dt # Start tracking date from the provided start_date

    # Add the starting pulls row if start_date is before the first event
    first_event_date = datetime.strptime(sorted(events_by_date.keys())[0], "%Y-%m-%d") if events_by_date else end_dt
    if start_dt <= first_event_date: # Use <= to include the case where start_date is the same as the first event
         rows.append({
            "Date": start_dt.date(),
            "Event": "Starting Pulls",
            "Jade Gain": 0,
            "Pull Gain": 0,
            "Cumulative Pulls": int(cumulative)
        })


    for date_str in sorted(events_by_date.keys()):
        event_date = datetime(int(date_str[:4]), int(date_str[5:7]), int(date_str[8:])) # Re-parse date for clarity

        # Calculate passive daily income from the day after the last processed date up to the current event date
        # NOTE: Weekly Passive Income entries are added as events. Their jade amount covers the passive for 7 days.
        # We only add passive income between events that are NOT weekly passive markers.
        days_since_last_calculation = (event_date - current_date).days
        if days_since_last_calculation > 0:
             passive_jade_between_events = days_since_last_calculation * daily_jade
             passive_pulls_between_events = passive_jade_between_events / jade_per_pull
             # Only add this passive income if the event is NOT a 'Weekly Passive Income' entry
             # The Weekly Passive Income entry already includes the passive for its period.
             if "Weekly Passive Income" not in events_by_date[date_str][0][0]: # Check if the FIRST event on this date is weekly passive
                 cumulative += passive_pulls_between_events


        # Process all events on the current date
        for name, jade in events_by_date[date_str]:
            # Event-specific pull gain
            if "Embers Exchange" in name:
                evt_pulls = 5
            elif "Express Pass Renewal" in name:
                evt_pulls = 300 / jade_per_pull
            elif "Weekly Passive Income" in name:
                 # The jade for the weekly passive entry represents the pulls for that 7-day period
                 # This jade amount is already calculated and added as the event's jade
                 # So we just use the jade to calculate pulls for the 'Pull Gain' column
                 evt_pulls = jade / jade_per_pull
            else:
                evt_pulls = jade / jade_per_pull

            # Add the event pulls to cumulative. For Weekly Passive Income, the cumulative update happens here
            # based on the evt_pulls calculated from the weekly_passive_jade_amount.
            cumulative += evt_pulls


            rows.append({
                "Date": event_date.date(),
                "Event": name,
                "Jade Gain": jade,
                "Pull Gain": round(evt_pulls, 2),
                "Cumulative Pulls": int(cumulative) # Truncate for display
            })

        # Update current_date to the current event date for the next iteration's passive calculation
        # It will be incremented by one day at the start of the next loop iteration's passive calculation
        current_date = event_date


    # Final remaining passive income
    if current_date < end_dt:
        days = (end_dt - current_date).days # Days from the last event date to the end_date
        if days > 0: # Only add if there are days after the last event
            passive_jade = days * daily_jade
            passive_pulls = passive_jade / jade_per_pull
            cumulative += passive_pulls
            # Add a final passive income row at the end_date
            rows.append({
                "Date": end_dt.date(),
                "Event": "Remaining Passive Income",
                "Jade Gain": passive_jade,
                "Pull Gain": round(passive_pulls, 2),
                "Cumulative Pulls": int(cumulative)
            })
    elif current_date == end_dt and not rows: # Handle case where end_date is start_date and no events
         # This case should be handled by the initial starting pulls row now
         pass # No action needed here if start_dt <= first_event_date block handles it.


    df = pd.DataFrame(rows)
    return df

## Pull Simulation Test

In [7]:
events = [
    ("2025-05-16", "Livestream", 300),
    ("2025-05-20", "Bug and update compensation", 300),
    ("2025-05-20", "Character Demos", 40),
    ("2025-05-20", "Galactic Baseballer Event", 1420),
    ("2025-05-20", "Character Demos", 80),
    ("2025-05-21", "Gift of Odyssey", 160),
    ("2025-05-22", "Gift of Odyssey", 160),
    ("2025-05-23", "Gift of Odyssey", 320),
    ("2025-05-24", "Gift of Odyssey", 160),
    ("2025-05-25", "Gift of Odyssey", 160),
    ("2025-05-26", "Gift of Odyssey", 160),
    ("2025-05-27", "Gift of Odyssey", 480),
    ("2025-06-11", "Trace and Drift Event", 500),
    ("2025-06-11", "Character Demos", 40),
    ("2025-06-27", "Livestream", 300),
    ("2025-06-30", "Divergent universe synchronicity + stable array estimated completion", 660),
    ("2025-07-01", "Bug and update compensation", 300),
    ("2025-07-01", "Character Demos", 80),
    ("2025-07-01", "Gift of Odyssey", 160),
    ("2025-07-02", "Gift of Odyssey", 160),
    ("2025-07-03", "Gift of Odyssey", 320),
    ("2025-07-04", "Gift of Odyssey", 160),
    ("2025-07-05", "Gift of Odyssey", 160),
    ("2025-07-06", "Gift of Odyssey", 160),
    ("2025-07-07", "Gift of Odyssey", 480)
]

In [5]:
import os
print(os.getcwd())

/content


In [8]:
exp_pass_renewal = "2025-07-01"

pull_df = generate_pull_projection(current_pulls=232, start_date="2025-06-16", end_date="2025-07-02", events=events, express_renewal_start=exp_pass_renewal, nameless_honor=True)
print(pull_df)
pull_df.to_excel('output.xlsx', index=False)

          Date                                              Event  Jade Gain  \
0   2025-06-16                                     Starting Pulls          0   
1   2025-06-16                   Simulated Universe Weekly Reward        225   
2   2025-06-19                      HoYoLAB Check-in (3rd Reward)         20   
3   2025-06-23                              Memory of Chaos Reset        800   
4   2025-06-23                   Simulated Universe Weekly Reward        225   
5   2025-06-23                     Weekly Passive Income (7 days)       1050   
6   2025-06-27                                         Livestream        300   
7   2025-06-30  Divergent universe synchronicity + stable arra...        660   
8   2025-06-30                   Simulated Universe Weekly Reward        225   
9   2025-06-30                     Weekly Passive Income (7 days)       1050   
10  2025-07-01                        Bug and update compensation        300   
11  2025-07-01                          